In [1]:
# Import the below libraries:
"""For faster loading, consider importing the libraries in separate cells."""
# To create reproducible file paths
import os # To interact with the operating system
from pathlib import Path # To create Path objects
import pathlib # To access the object-oriented library for paths

# For unzipping folders
import time # To handle time-related tasks
import zipfile # To download and extract zip files

# For saving Python objects
import pickle # To pickle datasets that take long periods to process

# To use APIs
import earthaccess # For logging into NASA Earth Access
import requests # For sending API requests

# To download Global Biodiversity Information Facility (GBIF) data
from getpass import getpass # To obtain a GBIF login and password
import pygbif.occurrences as occ # To download species occurrence data from GBIF
import pygbif.species as species # To identify specific species datasets to download on GBIF

# To work with different types of data
import geopandas as gpd # To make GeoDataFrames/work with vector data
from glob import glob # To combine data arrays
from math import floor, ceil # For dealing with integers
import numpy as np # To work with arrays
import pandas as pd # To work with dataframes
import rasterio # For handling raster soil maps
import rioxarray as rxr # To work with raster data
import rioxarray.merge as rxrm # To merge raster data
from shapely.geometry import MultiPolygon, Polygon # To handle invalid geometries
import xarray as xr # To use xarray datasets
import xrspatial # To handle spatial data

# For visualization and interactive plotting
import holoviews as hv # To create interactive hvPlots
import hvplot.pandas # To enable hvPlot interactive plotting for Pandas dataframes
import hvplot.xarray # To enable hvPlot interactive plotting for xarray datasets
import matplotlib.gridspec as gridspec # To display the plot legends properly
import matplotlib.pyplot as plt # To import the Pyplot module

In [2]:
# Create a designated folder for the GBIF and nutrient data
"""Note: the precise destination of the repository folder will vary from individual to individual."""
nutrient_data_dir = os.path.join(
    pathlib.Path.home(),
    # In the earth-analytics data folder
    'earth-analytics',
    'data',
    # Specify the destination as inside the "PLANT-B-Stoichiometric-Nutrient-Cycle-Simulation" repository
    'PLANT-B-Stoichiometric-Nutrient-Cycle-Simulation',
    'habitat_nutrient_concentrations_for_species'
)

# Create the directory
os.makedirs(nutrient_data_dir, exist_ok=True)

In [3]:
# Create a directory to store the GBIF data
gbif_dir = os.path.join(nutrient_data_dir, 'gbif_PLANT-B_dir')
os.makedirs(gbif_dir, exist_ok=True)

In [4]:
# Permanently log into GBIF:
"""A .env file was made to store the login information and avoid repeat requests. Only a redacted version will be published on GitHub  
to avoid sharing confidential information."""
# Do not reset credentials to avoid repeated login requests
reset_credentials = False

# Request and store username
if (not ('GBIF_USER'  in os.environ)) or reset_credentials:
    os.environ['GBIF_USER'] = input('GBIF username:')

# Request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset_credentials:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
# Request and store email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset_credentials:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

In [5]:
# Check that the login attempt was successful
'GBIF_PWD' in os.environ

True

In [6]:
# Set the species names for the white dwarf isopod, the temperate white springtail, and the spreading earthmoss:
# White dwarf isopod
isopod_name = "Trichorhina tomentosa"
# Temperate white springtail
springtail_name = "Folsomia candida"
# Spreading earthmoss
moss_name = "Physcomitrella patens"

# Pull the species info from GBIF
isopod_info = species.name_lookup(isopod_name, rank = 'Species')
springtail_info = species.name_lookup(springtail_name, rank = 'Species')
moss_info = species.name_lookup(moss_name, rank = 'Species')

# Grab the second result for the white dwarf isopod and print it
"""The second result matches the desired dataset, pulled from https://www.gbif.org/species/2204981."""
isopod_result = isopod_info['results'][1]
print("\nWhite dwarf isopod (Trichorhina tomentosa):")
print(isopod_result)

# Grab the result for the temperate white springtail and print it
springtail_result = springtail_info['results'][0]
print("\nTemperate white springtail (Folsomia candida):")
print(springtail_result)

# Grab the result for the spreading earthmoss and print it
moss_result = moss_info['results'][0]
print("\nSpreading earthmoss (Physcomitrella patens):")
print(moss_result)


White dwarf isopod (Trichorhina tomentosa):
{'key': 145956763, 'datasetKey': '3f5e930b-52a5-461d-87ec-26ecd66f14a3', 'nubKey': 2204981, 'parentKey': 315793644, 'parent': 'Trichorhina', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'genus': 'Trichorhina', 'species': 'Trichorhina tomentosa', 'kingdomKey': 315792181, 'phylumKey': 315793470, 'classKey': 315793535, 'genusKey': 315793644, 'speciesKey': 145956763, 'scientificName': 'Trichorhina tomentosa', 'canonicalName': 'Trichorhina tomentosa', 'nameType': 'SCIENTIFIC', 'taxonomicStatus': 'ACCEPTED', 'rank': 'SPECIES', 'origin': 'SOURCE', 'numDescendants': 0, 'numOccurrences': 0, 'taxonID': 'rinse-checklist:taxon:ab18f069b6b2207654fe78190efb3ef9', 'habitats': ['FRESHWATER'], 'nomenclaturalStatus': [], 'threatStatuses': [], 'descriptions': [], 'vernacularNames': [], 'synonym': False, 'higherClassificationMap': {'315792181': 'Animalia', '315793470': 'Arthropoda', '315793535': 'Malacostraca', '315793644': 'Trichorhina'}, 'class': 'Malacostr

In [7]:
# Get the species key for the dwarf white isopod
isopod_key = isopod_result['nubKey']

# Check what the isopod's species key is
isopod_name, isopod_key

('Trichorhina tomentosa', 2204981)

In [8]:
# Get the species key for the temperate white springtail
springtail_key = springtail_result['nubKey']

# Check what the springtail's species key is
springtail_name, springtail_key

('Folsomia candida', 2120383)

In [9]:
# Get the species key for the spreading earthmoss
moss_key = moss_result['nubKey']

# Check what the moss's species key is
moss_name, moss_key

('Physcomitrella patens', 2674024)

In [10]:
# Create a directory to store the isopod GBIF data
isopod_gbif_dir = os.path.join(gbif_dir, 'isopod_gbif_dir')
os.makedirs(isopod_gbif_dir, exist_ok=True)

# Create the file path for the isopod CSV
isopod_gbif_pattern = os.path.join(isopod_gbif_dir, '*.csv')

# Create a function to download the dwarf white isopod occurrence data
if not glob(isopod_gbif_pattern): # For the isopod data download, ensure the pattern used is "isopod_gbif_pattern"
    # Submit the query to GBIF
    gbif_query = occ.download([
        f"speciesKey = {isopod_key}", # For the isopod data download, ensure the key used is "isopod_key"
        "hasCoordinate = True", # For all datasets, only accept occurrence points that have longitude and latitude coordinates
    ])
    # Only download the data once
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]
        download_key = os.environ['GBIF_DOWNLOAD_KEY']
        time.sleep(5) # Let it rest before reattempting the download
    # Download the data
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'],
        path = isopod_gbif_dir
    )
    # Unzip the file
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = isopod_gbif_dir)

# Locate the CSV file path for the isopod data download
isopod_gbif_path = glob(isopod_gbif_pattern)[0]

INFO:Your download key is 0008929-260409193756587
INFO:Download file size: 12710 bytes
INFO:On disk at C:\Users\livth\earth-analytics\data\PLANT-B-Stoichiometric-Nutrient-Cycle-Simulation\habitat_nutrient_concentrations_for_species\gbif_PLANT-B_dir\isopod_gbif_dir/0008929-260409193756587.zip


In [11]:
# Look at the isopod download information
occ.download_meta("0008929-260409193756587") # Input the download key to view the information

{'key': '0008929-260409193756587',
 'doi': '10.15468/dl.trbbxx',
 'license': 'http://creativecommons.org/licenses/by-nc/4.0/legalcode',
 'request': {'predicate': {'type': 'and',
   'predicates': [{'type': 'equals',
     'key': 'SPECIES_KEY',
     'value': '2204981',
     'matchCase': False},
    {'type': 'equals',
     'key': 'HAS_COORDINATE',
     'value': 'True',
     'matchCase': False}]},
  'sendNotification': True,
  'format': 'SIMPLE_CSV',
  'type': 'OCCURRENCE',
  'verbatimExtensions': [],
  'interpretedExtensions': []},
 'created': '2026-04-12T23:40:56.768+00:00',
 'modified': '2026-04-12T23:42:34.715+00:00',
 'eraseAfter': '2026-10-12T23:40:56.642+00:00',
 'status': 'SUCCEEDED',
 'downloadLink': 'https://api.gbif.org/v1/occurrence/download/request/0008929-260409193756587.zip',
 'size': 12710,
 'totalRecords': 136,
 'numberDatasets': 27}

In [12]:
# Create a directory to store the springtail GBIF data
springtail_gbif_dir = os.path.join(gbif_dir, 'springtail_gbif_dir')
os.makedirs(springtail_gbif_dir, exist_ok=True)

# Create the file path for the springtail CSV
springtail_gbif_pattern = os.path.join(springtail_gbif_dir, '*.csv')

# Create a function to download the temperate white springtail occurrence data
if not glob(springtail_gbif_pattern): # For the springtail data download, ensure the pattern used is "springtail_gbif_pattern"
    # Submit the query to GBIF
    gbif_query = occ.download([
        f"speciesKey = {springtail_key}", # For the springtail data download, ensure the key used is "springtail_key"
        "hasCoordinate = True", # For all datasets, only accept occurrence points that have longitude and latitude coordinates
    ])
    # Only download the data once
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]
        download_key = os.environ['GBIF_DOWNLOAD_KEY']
        time.sleep(5) # Let it rest before reattempting the download
    # Download the data
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'],
        path = springtail_gbif_dir
    )
    # Unzip the file
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = springtail_gbif_dir)

# Locate the CSV file path for the springtail data download
springtail_gbif_path = glob(springtail_gbif_pattern)[0]

In [13]:
# Look at the springtail download information
occ.download_meta("0009002-260409193756587") # Input the download key to view the information

{'key': '0009002-260409193756587',
 'doi': '10.15468/dl.yskvve',
 'license': 'http://creativecommons.org/licenses/by-nc/4.0/legalcode',
 'request': {'predicate': {'type': 'and',
   'predicates': [{'type': 'equals',
     'key': 'SPECIES_KEY',
     'value': '2120383',
     'matchCase': False},
    {'type': 'equals',
     'key': 'HAS_COORDINATE',
     'value': 'True',
     'matchCase': False}]},
  'sendNotification': True,
  'format': 'SIMPLE_CSV',
  'type': 'OCCURRENCE',
  'verbatimExtensions': [],
  'interpretedExtensions': []},
 'created': '2026-04-13T00:13:09.896+00:00',
 'modified': '2026-04-13T00:14:17.693+00:00',
 'eraseAfter': '2026-10-13T00:13:09.710+00:00',
 'status': 'SUCCEEDED',
 'downloadLink': 'https://api.gbif.org/v1/occurrence/download/request/0009002-260409193756587.zip',
 'size': 61278,
 'totalRecords': 1388,
 'numberDatasets': 101}

In [14]:
# Create a directory to store the moss GBIF data
moss_gbif_dir = os.path.join(gbif_dir, 'moss_gbif_dir')
os.makedirs(moss_gbif_dir, exist_ok=True)

# Create the file path for the moss CSV
moss_gbif_pattern = os.path.join(moss_gbif_dir, '*.csv')

# Create a function to download the spreading earthmoss occurrence data
if not glob(moss_gbif_pattern): # For the moss data download, ensure the pattern used is "moss_gbif_pattern"
    # Submit the query to GBIF
    gbif_query = occ.download([
        f"speciesKey = {moss_key}", # For the moss data download, ensure the key used is "moss_key"
        "hasCoordinate = True", # For all datasets, only accept occurrence points that have longitude and latitude coordinates
    ])
    # Only download the data once
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]
        download_key = os.environ['GBIF_DOWNLOAD_KEY']
        time.sleep(5) # Let it rest before reattempting the download
    # Download the data
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'],
        path = moss_gbif_dir
    )
    # Unzip the file
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = moss_gbif_dir)

# Locate the CSV file path for the moss data download
moss_gbif_path = glob(moss_gbif_pattern)[0]

INFO:Your download key is 0009040-260409193756587
INFO:Download file size: 12710 bytes
INFO:On disk at C:\Users\livth\earth-analytics\data\PLANT-B-Stoichiometric-Nutrient-Cycle-Simulation\habitat_nutrient_concentrations_for_species\gbif_PLANT-B_dir\moss_gbif_dir/0008929-260409193756587.zip


In [15]:
# Look at the moss download information
occ.download_meta("0009040-260409193756587") # Input the download key to view the information

{'key': '0009040-260409193756587',
 'doi': '10.15468/dl.7gxzry',
 'license': 'unspecified',
 'request': {'predicate': {'type': 'and',
   'predicates': [{'type': 'equals',
     'key': 'SPECIES_KEY',
     'value': '2674024',
     'matchCase': False},
    {'type': 'equals',
     'key': 'HAS_COORDINATE',
     'value': 'True',
     'matchCase': False}]},
  'sendNotification': True,
  'format': 'SIMPLE_CSV',
  'type': 'OCCURRENCE',
  'verbatimExtensions': [],
  'interpretedExtensions': []},
 'created': '2026-04-13T00:24:31.544+00:00',
 'modified': '2026-04-13T00:25:53.963+00:00',
 'eraseAfter': '2026-10-13T00:24:31.403+00:00',
 'status': 'SUCCEEDED',
 'downloadLink': 'https://api.gbif.org/v1/occurrence/download/request/0009040-260409193756587.zip',
 'size': 516,
 'totalRecords': 0,
 'numberDatasets': 0}